In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests

from pyjstat import pyjstat
from collections import OrderedDict

In [2]:
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany" : "DE",
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "EL",   ## NEW
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Cyprus" : "CY",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "UK",  ## NEW
    "Iceland" : "IS",
    "Norway" : "NO",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Albania" : "AL",
    "Serbia" : "RS",
    "Türkiye" : "TR",                     # Leon edit, corrected country name as used in the dataset 
    "Bosnia and Herzegovina" : "BA",
    "Kosovo (under United Nations Security Council Resolution 1244/99)" : "XK",
    "Moldova" : "MD",
    "Ukraine" : "UA",
    "Georgia" : "GE",
    "Liechtenstein" : "LI"                # Leon edit, added Liechtenstein 
}

print(list(map_country_ISO.values()))

['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE', 'UK', 'IS', 'NO', 'ME', 'MK', 'AL', 'RS', 'TR', 'BA', 'XK', 'MD', 'UA', 'GE', 'LI']


In [4]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/query-builder/tool
indicator = 'nrg_pc_203'
dataformat = 'JSON'

params = dict(
    sinceTimePeriod = '2020-S1',
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
    unit = 'KWH',
    product = '4100',
    nrg_cons='TOT_GJ',
    tax = 'X_VAT',
    currency = 'EUR',
    lang = 'en'
)

In [5]:
#print(", ".join(params["geo"]))

In [6]:
#print(set(params["geo"]) - set(map_country_ISO.values()))
#print(set(map_country_ISO.values()) - set(params["geo"]))

In [7]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"+indicator+"?format="+dataformat
r = requests.get(url=url, params=params)
nrg_pc_203_temp = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])
nrg_pc_203_temp["Geopolitical entity (reporting)"].unique()


array(['Belgium', 'Bulgaria', 'Czechia', 'Denmark', 'Germany', 'Estonia',
       'Ireland', 'Greece', 'Spain', 'France', 'Croatia', 'Italy',
       'Latvia', 'Lithuania', 'Luxembourg', 'Hungary', 'Netherlands',
       'Austria', 'Poland', 'Portugal', 'Romania', 'Slovenia', 'Slovakia',
       'Finland', 'Sweden', 'Liechtenstein', 'United Kingdom',
       'Bosnia and Herzegovina', 'Moldova', 'North Macedonia', 'Georgia',
       'Albania', 'Serbia', 'Türkiye', 'Ukraine'], dtype=object)

In [8]:
# Leon edit, angefragte vs. erhaltene Rohdaten (= nrg_pc_203_temp) von eurostat: 

# 1. Angefragte Länder (ISO-Codes)
#angefragt = set(params["geo"])

# 2. Gelieferte Länder (Langnamen) aus dem DataFrame
#geliefert_namen = set(nrg_pc_203_temp["Geopolitical entity (reporting)"].unique())

# 3. Länder mit Daten → umwandeln in ISO-Codes
#geliefert_iso = set(map_country_ISO.get(name) for name in geliefert_namen if name in map_country_ISO)

# 4. ISO-Codes, die angefragt wurden, aber nicht geliefert wurden
#fehlende_iso = angefragt - geliefert_iso

# 5. Die zugehörigen Ländernamen aus map_country_ISO (umgedreht)
#fehlende_namen = [name for name, code in map_country_ISO.items() if code in fehlende_iso]

#print("Nicht gelieferte Ländernamen:")
#print(fehlende_namen)

In [9]:
nrg_pc_203_temp["Geopolitical entity (reporting)"].unique()

array(['Belgium', 'Bulgaria', 'Czechia', 'Denmark', 'Germany', 'Estonia',
       'Ireland', 'Greece', 'Spain', 'France', 'Croatia', 'Italy',
       'Latvia', 'Lithuania', 'Luxembourg', 'Hungary', 'Netherlands',
       'Austria', 'Poland', 'Portugal', 'Romania', 'Slovenia', 'Slovakia',
       'Finland', 'Sweden', 'Liechtenstein', 'United Kingdom',
       'Bosnia and Herzegovina', 'Moldova', 'North Macedonia', 'Georgia',
       'Albania', 'Serbia', 'Türkiye', 'Ukraine'], dtype=object)

In [10]:
nrg_pc_203_temp["Geopolitical entity (reporting)"].head()

0    Belgium
1    Belgium
2    Belgium
3    Belgium
4    Belgium
Name: Geopolitical entity (reporting), dtype: object

In [11]:
#rename countries and convert to MWh
nrg_pc_203 = nrg_pc_203_temp.copy().dropna()
nrg_pc_203 = nrg_pc_203.rename(columns={'Geopolitical entity (reporting)':'country'})
nrg_pc_203['country'] = nrg_pc_203['country'].map(map_country_ISO)
nrg_pc_203['year'] = nrg_pc_203['Time'].str[:4]
nrg_pc_203 = nrg_pc_203[['country', 'year', 'value']].dropna() #need to check where the NaNs come from
nrg_pc_203 = nrg_pc_203.groupby(['country','year']).mean()
nrg_pc_203['EUR_per_MWh'] = nrg_pc_203['value'] * 1000
nrg_pc_203 = nrg_pc_203[['EUR_per_MWh']]
nrg_pc_203.head()

#print(nrg_pc_203['country'].unique())

EUR_per_MWh
country year             
AT      2021        48.70
        2022        87.35
        2023        63.75
        2024        51.85
BA      2021        38.70

In [12]:
# Leon - consistency check: 
#print(set(nrg_pc_203['country']) - set(map_country_ISO.values()))
#print(set(map_country_ISO.values()) - set(nrg_pc_203['country']))
#print(set(nrg_pc_203['country']) - set(params["geo"]))
#print(set(params["geo"]) - set(nrg_pc_203['country']))

In [13]:
# Leon edit, erhaltene Rohdaten (=nrg_pc_203_temp) vs. mit dropna() bearbeite Daten:

#fehlende = set(nrg_pc_203_temp["Geopolitical entity (reporting)"])-set(nrg_pc_203["Geopolitical entity (reporting)"])
#print(fehlende)

In [14]:
# Leon edit, Überprüfung bei den Rohdaten (= nrg_pc_203_temp), dass die unter dropna() weggefallenen Länder durchgehend bei value NaN aufweisen

# am Beispiel: Luxembourg: 
#Missing_Country = nrg_pc_203_temp[nrg_pc_203_temp["Geopolitical entity (reporting)"] == "Luxembourg"]
#Missing_Country

In [15]:
nrg_pc_203.reset_index().country.unique()

array(['AT', 'BA', 'BE', 'BG', 'CZ', 'DE', 'EE', 'EL', 'ES', 'FI', 'FR',
       'HR', 'HU', 'IT', 'MK', 'NL', 'PL', 'PT', 'RO', 'RS', 'SK', 'TR'],
      dtype=object)

In [170]:
nrg_pc_203.to_csv(dir_out+'price_gas_yearly_Eurostat.csv', encoding="utf-8")

OSError: Cannot save file into a non-existent directory: '..\parsed_data'